# CourtLens GPU eval (generated by scripts/kaggle/build_kaggle.py — do not edit here)
Runs the real backend pipeline (`app.scripts.run_offline`) on facility CCTV, DFL clips, and a synthetic 3-camera check. Accelerator: **GPU T4**.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile
from pathlib import Path

# System deps: ffmpeg (probe/sync), tesseract (jersey OCR — was missing in old runs)
subprocess.call("which ffmpeg ffprobe || (apt-get -qq update && apt-get -qq install -y ffmpeg)", shell=True)
subprocess.call("which tesseract || (apt-get -qq update && apt-get -qq install -y tesseract-ocr)", shell=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *['ultralytics==8.4.183', 'lap>=0.5.12', 'pydantic-settings==2.6.1', 'pytesseract>=0.3.10', 'torchreid==0.2.5', 'umap-learn>=0.5.6', 'motmetrics>=1.4.0', 'remotezip>=0.12']])

# Kaggle auto-extracts uploaded zips; accept either the zip or the extracted tree
BACKEND = Path("/kaggle/working/backend")
if BACKEND.exists():
    shutil.rmtree(BACKEND)
code_zip = glob.glob("/kaggle/input/**/courtlens_backend.zip", recursive=True)
runner = glob.glob("/kaggle/input/**/app/scripts/run_offline.py", recursive=True)
if code_zip:
    zipfile.ZipFile(code_zip[0]).extractall(BACKEND)
elif runner:
    shutil.copytree(Path(runner[0]).parents[2], BACKEND)  # read-only input → writable copy
else:
    raise SystemExit("attach dataset muddasirr/courtlens-code")
print("code rev", (BACKEND / "GIT_REV").read_text())
mounted = (BACKEND / "BUILD_ID").read_text().strip() if (BACKEND / "BUILD_ID").exists() else "none"
assert mounted == "20261023-211534", f"stale code bundle {mounted} (expected 20261023-211534) — re-push after the dataset finishes processing"

In [ ]:
import torch
assert torch.cuda.is_available(), "Settings → Accelerator → GPU T4, then re-run"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import json
env = dict(os.environ, PYTHONPATH=str(BACKEND), VISION_DEVICE="0", VISION_TARGET_FPS="10", YOLO_MODEL="yolov8m.pt")
OUT = Path("/kaggle/working/eval"); OUT.mkdir(exist_ok=True)

In [ ]:
# 6) Detection + tracking vs SoccerTrack v2 MOT ground truth (4096x1080 fixed panorama clips)
mot_runs = []
# Kaggle extracts the upload without the mot/ prefix: <id>.txt beside clips/<id>.mp4
for gt in sorted(Path("/kaggle/input").rglob("*.txt")):
    clip = gt.parent / "clips" / f"{gt.stem}.mp4"
    if not clip.exists():
        continue
    out_json = OUT / f"bench_mot_{gt.stem}.json"
    r = subprocess.run([sys.executable, "-m", "app.scripts.bench_mot", "--clip", str(clip), "--gt", str(gt),
                        "--imgsz", os.environ.get("COURTLENS_MOT_IMGSZ", "1920"),
                        "--max-frames", "600", "--out", str(out_json), "--reid", "--save-log", str(OUT / "mot_logs"),
                        *(["--teams", str(t)] if (t := BACKEND / "app/scripts/bench_labels" / f"{gt.stem}_teams.json").exists() else [])],
                       cwd=BACKEND, env=env, capture_output=True, text=True)
    (OUT / f"bench_mot_{gt.stem}.log").write_text(r.stdout + "\n" + r.stderr)
    print(r.stdout[-2000:])

In [ ]:
# 5) One-file bundles — `kaggle kernels output` fetches files one by one (thousands of thumbs)
import zipfile
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file() and (p.suffix in {".json", ".log"} or p.name == "jersey_resnet18.pt"):
            zf.write(p, p.relative_to(OUT.parent))
with zipfile.ZipFile("/kaggle/working/thumbs.zip", "w", zipfile.ZIP_STORED) as zf:
    for p in OUT.rglob("thumbs/*/*.jpg"):
        zf.write(p, p.relative_to(OUT.parent))
print("bundles", Path("/kaggle/working/results.zip").stat().st_size // 1_000_000, "MB")